## Producer B — Kafka Event Stream Initialisation

This cell initialises and runs **Producer B**, which reads camera events from `camera_event_B.csv`
and publishes them to the Kafka topic `camera-events-B` in batches grouped by `batch_id`.

### Key Parameters

| Parameter | Value | Rationale |
|---|---|---|
| `bootstrap_servers` | `kafka:9092` | Kafka broker address as defined in Docker Compose network. Using the service name `kafka` instead of `localhost` ensures correct container-to-container routing. |
| `topic` | `camera-events-B` | Dedicated topic per producer isolates each camera stream, allowing the Spark consumer to subscribe selectively and apply per-stream logic. |
| `camera_id` | `2` | Identifies the source camera for all events published by this producer, enabling traceability in the downstream violation detection logic. |
| `batch_interval` | `5 seconds` | Events are grouped by `batch_id` and published one batch every 5 seconds. This pacing was confirmed as appropriate by the teaching team. It provides a realistic simulation of a camera emitting periodic snapshots, while giving Spark sufficient time to process each micro-batch before the next arrives. |
| `csv_path` | `../data/camera_event_B.csv` | Relative path to the camera event dataset for Producer B, following the submission directory structure defined in the specification. |

### Execution Notes

- The producer runs continuously until manually interrupted (`KeyboardInterrupt`).
- On interrupt or error, `producer_b.close()` is called in the `finally` block to ensure the Kafka
  producer is gracefully shut down and all buffered messages are flushed.
- Run this notebook **concurrently** with Producer A and Producer C notebooks to simulate
  simultaneous multi-camera event ingestion, as required by the streaming join logic in Task 2.1.2.

In [ ]:
from pathlib import Path

from camera_event_producer import CameraEventProducer

HOST_IP = "kafka"  # Docker Compose service name
csv_path = Path("..") / "data" / "camera_event_B.csv"
producer_b = CameraEventProducer(
    bootstrap_servers=[f"{HOST_IP}:9092"],
    topic="camera-events-B",
    camera_id=2,
    csv_path=str(csv_path),
    batch_interval=5
)

try:
    producer_b.publish_batches()
except KeyboardInterrupt:
    print("Producer B stopped by user")
finally:
    producer_b.close()

[2026-05-25T12:36:31.518082] Published batch 1 to camera-events-B (5 events)
[2026-05-25T12:36:36.537807] Published batch 2 to camera-events-B (1 events)
[2026-05-25T12:36:41.547392] Published batch 3 to camera-events-B (1 events)
[2026-05-25T12:36:46.557570] Published batch 4 to camera-events-B (4 events)
[2026-05-25T12:36:51.567470] Published batch 5 to camera-events-B (4 events)
[2026-05-25T12:36:56.578076] Published batch 6 to camera-events-B (2 events)
[2026-05-25T12:37:01.593557] Published batch 7 to camera-events-B (3 events)
[2026-05-25T12:37:06.607721] Published batch 8 to camera-events-B (1 events)
[2026-05-25T12:37:11.620207] Published batch 9 to camera-events-B (2 events)
[2026-05-25T12:37:16.628164] Published batch 10 to camera-events-B (2 events)
[2026-05-25T12:37:21.641484] Published batch 11 to camera-events-B (1 events)
[2026-05-25T12:37:26.654526] Published batch 12 to camera-events-B (1 events)
[2026-05-25T12:37:31.675656] Published batch 13 to camera-events-B (1 eve